[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/09.%20The%20Quay%20Crane%20Scheduling%20Problem/P9-Tier-5_executed.ipynb)

---


# 9. The Quay Crane Scheduling Problem
## Tier 5 — Integrated Digital Twin (System-of-Systems Simulation)

This notebook builds a small, time-stepped terminal simulation. It compares a **baseline assignment rule** with a **priority-based coordination heuristic**, then examines how both behave under three specified disruptions. The simulation is a teaching model: it illustrates how quay, yard, vessel, and gate measures can be tracked together, but it is not a calibrated operational Digital Twin or a learned dispatcher.

### Learning goals
- Understand how a discrete-time simulation represents the changing state of interconnected terminal subsystems.
- Follow the crane-assignment rules and see how they affect simulated vessel progress.
- Aggregate time-series observations into operational key performance indicators (KPIs).
- Compare baseline and coordinated-heuristic results without treating a simulation as proof of real-world improvement.
- Explain what the disruption scenarios represent and what their simplified assumptions leave out.

### Inputs and outputs
The input instance contains six quay cranes, eighteen yard cranes, two vessels, a seven-day horizon, and 15-minute time steps. Crane productivities and truck arrivals include stochastic variation; the random seed is fixed for reproducibility. The notebook outputs KPI tables, comparison plots, time-series plots, and scenario summaries when the cells are run.

### Scope and modeling assumptions
A vessel can receive quay cranes only after its stated arrival time. Each vessel can have at most three cranes assigned at once, and a vessel's processed volume cannot exceed its workload. Crane interference is represented by a multiplicative productivity factor. Yard activity is represented by a capacity-based utilization proxy, and gate activity by a single queue with fixed processing capacity.

The model does not represent berth geometry, bay-level crane conflicts, container-level flows, yard queues, truck appointments, weather-dependent productivity, or operating costs. The two assignment rules are simple heuristics; the coordinated rule is not an AI model or an optimization solver. Consequently, outputs are illustrative results of the stated assumptions, not forecasts or evidence of annual savings.

In [ ]:
# Dependencies are expected to be available in the Python environment.
try:
    import random
    from dataclasses import dataclass, field
    from typing import Any, Dict, List, Set
    from datetime import datetime, timedelta

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
except ImportError as error:
    raise ImportError(
        "Missing dependency. Install numpy, pandas, and matplotlib in your environment."
    ) from error

print("Dependencies imported successfully.")

## Digital Twin architecture overview

The conceptual Digital Twin has five connected modules:

1. **Quay Operations Module**: represents crane assignments, processing rates, and interference.
2. **Vessel Intelligence Module**: represents vessel arrival times, workloads, and service priorities.
3. **Yard Coordination Module**: represents yard-crane capacity and an aggregate utilization measure.
4. **Gate Operations Module**: represents stochastic truck arrivals, a queue, and gate processing capacity.
5. **Terminal Control System**: applies the assignment rule and advances the shared simulation state.

A **state** is the information needed to describe the simulated system at one point in time: the current clock, vessel progress, crane assignments, queue length, and recorded KPI history. A time step applies the operating rules to that state and produces the next observations. Here, each step is 15 minutes, so a seven-day run contains 672 steps. This is a discrete-time simulation, not a continuous representation of every real-world event.

In [ ]:
@dataclass(frozen=True)
class Vessel:
    id: str
    arrival_time: datetime
    teu_total: int
    bay_count: int
    priority: float


@dataclass(frozen=True)
class QuayCrane:
    id: int
    productivity_base: float  # TEU per hour
    interference_factor: float


@dataclass(frozen=True)
class YardCrane:
    id: int
    productivity: float  # moves per hour
    block_id: int


@dataclass
class DigitalTwinState:
    current_time: datetime
    vessels: List[Vessel] = field(default_factory=list)
    quay_cranes: List[QuayCrane] = field(default_factory=list)
    yard_cranes: List[YardCrane] = field(default_factory=list)
    vessel_progress: Dict[str, float] = field(default_factory=dict)
    crane_assignments: Dict[int, str] = field(default_factory=dict)
    yard_utilization: Dict[int, float] = field(default_factory=dict)
    gate_queue_length: int = 0
    vessel_finish_times: Dict[str, datetime] = field(default_factory=dict)
    kpi_history: List[Dict[str, Any]] = field(default_factory=list)


# Simulation horizon and temporal resolution.
SIM_START = datetime(2025, 6, 23, 0, 0, 0)
SIM_DURATION_DAYS = 7
TIME_STEP = timedelta(minutes=15)
STEP_HOURS = TIME_STEP.total_seconds() / 3600

# Fix the seed before drawing equipment productivities so the instance is reproducible.
random.seed(42)
np.random.seed(42)

QUAY_CRANES = [
    QuayCrane(
        id=i,
        productivity_base=25.0 + random.uniform(-2, 2),
        interference_factor=0.08,
    )
    for i in range(6)
]
YARD_CRANES = [
    YardCrane(
        id=i,
        productivity=40.0 + random.uniform(-3, 3),
        block_id=i % 8,
    )
    for i in range(18)
]

VESSELS = [
    Vessel(
        id="V1",
        arrival_time=SIM_START + timedelta(hours=6),
        teu_total=2200,
        bay_count=22,
        priority=1.0,
    ),
    Vessel(
        id="V2",
        arrival_time=SIM_START + timedelta(days=2, hours=12),
        teu_total=1800,
        bay_count=18,
        priority=0.9,
    ),
]

print(
    f"Instance: {len(QUAY_CRANES)} quay cranes, "
    f"{len(YARD_CRANES)} yard cranes, {len(VESSELS)} vessels."
)

## Step 1 — Digital Twin simulation engine

At each time step, the simulation first assigns eligible cranes, then calculates vessel processing, updates aggregate yard and gate measures, and records KPIs. The baseline rule fills available crane positions in vessel-list order. The coordinated heuristic orders unfinished, arrived vessels by priority and remaining workload, and assigns up to three cranes to each; among available cranes, it prefers higher base productivity.

The model's principal processing relation is

$$
p_{c,v}(t) = b_c(1-\alpha_c)^{n_v(t)-1}, \qquad
q_v(t) = \min\left(r_v(t),\; \Delta t\sum_{c\in C_v(t)}p_{c,v}(t)\right).
$$

Here, $b_c$ is crane $c$'s base productivity in TEU per hour, $\alpha_c$ is its interference factor, $n_v(t)$ is the number of cranes assigned to vessel $v$, $C_v(t)$ is that assigned-crane set, $r_v(t)$ is the vessel's remaining workload in TEU, and $\Delta t$ is the time-step length in hours. The cap by $r_v(t)$ prevents simulated processing from exceeding the vessel's workload. The code implements these relations in `effective_crane_productivity` and `simulate_step`.

In [ ]:
def effective_crane_productivity(
    crane: QuayCrane,
    vessel_id: str,
    state: DigitalTwinState,
) -> float:
    """Return hourly crane productivity after the vessel's interference penalty."""
    co_assigned = sum(
        assigned_vessel_id == vessel_id
        for assigned_vessel_id in state.crane_assignments.values()
    )
    if co_assigned == 0:
        return 0.0
    return crane.productivity_base * (
        1 - crane.interference_factor
    ) ** (co_assigned - 1)


def assign_cranes_baseline(
    state: DigitalTwinState,
    unavailable_cranes: Set[int] | None = None,
) -> None:
    """Assign available cranes in vessel-list order, up to three per vessel."""
    unavailable_cranes = unavailable_cranes or set()
    available_cranes = [
        crane.id
        for crane in state.quay_cranes
        if crane.id not in state.crane_assignments
        and crane.id not in unavailable_cranes
    ]
    for vessel in state.vessels:
        arrived = state.current_time >= vessel.arrival_time
        unfinished = state.vessel_progress.get(vessel.id, 0.0) < vessel.teu_total
        if not arrived or not unfinished:
            continue
        assigned_count = sum(
            assigned_vessel_id == vessel.id
            for assigned_vessel_id in state.crane_assignments.values()
        )
        while available_cranes and assigned_count < 3:
            crane_id = available_cranes.pop(0)
            state.crane_assignments[crane_id] = vessel.id
            assigned_count += 1


def assign_cranes_optimized(
    state: DigitalTwinState,
    unavailable_cranes: Set[int] | None = None,
) -> None:
    """Assign cranes by priority and remaining workload using a coordination heuristic."""
    unavailable_cranes = unavailable_cranes or set()
    available_cranes = [
        crane.id
        for crane in state.quay_cranes
        if crane.id not in state.crane_assignments
        and crane.id not in unavailable_cranes
    ]
    eligible_vessels = [
        vessel
        for vessel in state.vessels
        if state.current_time >= vessel.arrival_time
        and state.vessel_progress.get(vessel.id, 0.0) < vessel.teu_total
    ]
    vessels_by_priority = sorted(
        eligible_vessels,
        key=lambda vessel: (
            vessel.priority,
            vessel.teu_total - state.vessel_progress.get(vessel.id, 0.0),
        ),
        reverse=True,
    )
    crane_by_id = {crane.id: crane for crane in state.quay_cranes}
    for vessel in vessels_by_priority:
        assigned_count = sum(
            assigned_vessel_id == vessel.id
            for assigned_vessel_id in state.crane_assignments.values()
        )
        sorted_cranes = sorted(
            available_cranes,
            key=lambda crane_id: crane_by_id[crane_id].productivity_base,
            reverse=True,
        )
        for crane_id in sorted_cranes[: max(0, 3 - assigned_count)]:
            state.crane_assignments[crane_id] = vessel.id
            available_cranes.remove(crane_id)


def simulate_step(
    state: DigitalTwinState,
    optimized: bool = False,
    unavailable_cranes: Set[int] | None = None,
    truck_variability: bool = False,
) -> None:
    """Advance the terminal state and record one 15-minute observation."""
    unavailable_cranes = unavailable_cranes or set()

    # Remove failed equipment before assignment so it cannot process work in this step.
    for crane_id in unavailable_cranes:
        state.crane_assignments.pop(crane_id, None)

    if optimized:
        assign_cranes_optimized(state, unavailable_cranes)
    else:
        assign_cranes_baseline(state, unavailable_cranes)

    crane_by_id = {crane.id: crane for crane in state.quay_cranes}
    vessel_by_id = {vessel.id: vessel for vessel in state.vessels}
    assigned_by_vessel: Dict[str, List[int]] = {}
    for crane_id, vessel_id in state.crane_assignments.items():
        assigned_by_vessel.setdefault(vessel_id, []).append(crane_id)

    processed_this_step = 0.0
    for vessel_id, crane_ids in assigned_by_vessel.items():
        vessel = vessel_by_id[vessel_id]
        remaining = vessel.teu_total - state.vessel_progress.get(vessel_id, 0.0)
        if remaining <= 0:
            continue
        hourly_capacity = sum(
            effective_crane_productivity(crane_by_id[crane_id], vessel_id, state)
            for crane_id in crane_ids
        )
        vessel_processed = min(remaining, hourly_capacity * STEP_HOURS)
        state.vessel_progress[vessel_id] = (
            state.vessel_progress.get(vessel_id, 0.0) + vessel_processed
        )
        processed_this_step += vessel_processed
        if state.vessel_progress[vessel_id] >= vessel.teu_total:
            state.vessel_progress[vessel_id] = float(vessel.teu_total)
            state.vessel_finish_times[vessel_id] = state.current_time + TIME_STEP
            for crane_id in crane_ids:
                state.crane_assignments.pop(crane_id, None)

    # Yard utilization is a capacity-based proxy for this step, not a yard queue model.
    yard_moves_needed = processed_this_step * 1.5
    total_yard_capacity = sum(
        yard_crane.productivity * STEP_HOURS for yard_crane in state.yard_cranes
    )
    step_yard_utilization = (
        min(1.0, yard_moves_needed / total_yard_capacity)
        if total_yard_capacity > 0
        else 0.0
    )
    block_ids = {yard_crane.block_id for yard_crane in state.yard_cranes}
    state.yard_utilization = {
        block_id: step_yard_utilization for block_id in block_ids
    }

    # Truck arrivals are stochastic; the disruption changes their spread, not their mean.
    mean_trucks = 180 * STEP_HOURS
    if truck_variability:
        new_trucks = max(
            0,
            int(np.random.normal(mean_trucks, 2.0 * np.sqrt(mean_trucks))),
        )
    else:
        new_trucks = int(np.random.poisson(mean_trucks))
    gate_capacity = 200 * STEP_HOURS
    processed_trucks = min(gate_capacity, state.gate_queue_length + new_trucks)
    state.gate_queue_length = max(
        0, state.gate_queue_length + new_trucks - processed_trucks
    )

    total_teu_processed = sum(state.vessel_progress.values())
    state.kpi_history.append(
        {
            "timestamp": state.current_time + TIME_STEP,
            "total_teu_processed": total_teu_processed,
            "active_cranes": len(state.crane_assignments),
            "avg_yard_utilization": (
                float(np.mean(list(state.yard_utilization.values())))
                if state.yard_utilization
                else 0.0
            ),
            "gate_queue_trucks": state.gate_queue_length,
        }
    )


def run_simulation(optimized: bool = False, seed: int = 42) -> DigitalTwinState:
    """Run the seven-day simulation using the selected assignment rule."""
    random.seed(seed)
    np.random.seed(seed)
    state = DigitalTwinState(
        current_time=SIM_START,
        vessels=VESSELS.copy(),
        quay_cranes=QUAY_CRANES.copy(),
        yard_cranes=YARD_CRANES.copy(),
    )
    steps = int(
        timedelta(days=SIM_DURATION_DAYS).total_seconds()
        / TIME_STEP.total_seconds()
    )
    for _ in range(steps):
        simulate_step(state, optimized=optimized)
        state.current_time += TIME_STEP
    return state


baseline_state = run_simulation(optimized=False, seed=42)
optimized_state = run_simulation(optimized=True, seed=42)
print("Baseline and coordinated-heuristic simulations completed.")

## Step 2 — Extract and compare KPIs

A KPI is a summary measure used to describe one aspect of system performance. The simulation records observations over time; `summarize_kpis` converts those observations into one value per measure. Crane utilization is the average number of assigned cranes divided by the number available. Yard utilization is the mean of the step-level capacity proxy. Gate queue is reported as the mean number of trucks waiting, not as waiting time. Vessel turnaround is measured from each vessel's arrival to its recorded completion; vessels that do not finish within the horizon are excluded from that average.

These definitions matter when interpreting comparisons: throughput and utilization are generally better when higher, while queue size and turnaround are generally better when lower.

In [ ]:
def summarize_kpis(state: DigitalTwinState) -> Dict[str, float]:
    """Aggregate recorded observations and completed-vessel turnaround times."""
    df = pd.DataFrame(state.kpi_history)
    if df.empty:
        return {}

    average_crane_utilization = (
        df["active_cranes"].mean() / len(state.quay_cranes)
        if state.quay_cranes
        else 0.0
    )
    turnaround_hours = [
        (state.vessel_finish_times[vessel.id] - vessel.arrival_time).total_seconds()
        / 3600
        for vessel in state.vessels
        if vessel.id in state.vessel_finish_times
    ]

    return {
        "total_teu_processed": float(df["total_teu_processed"].iloc[-1]),
        "avg_crane_utilization": float(average_crane_utilization),
        "avg_yard_utilization": float(df["avg_yard_utilization"].mean()),
        "avg_gate_queue_trucks": float(df["gate_queue_trucks"].mean()),
        "avg_vessel_turnaround_hours": (
            float(np.mean(turnaround_hours)) if turnaround_hours else 0.0
        ),
    }


baseline_kpis = summarize_kpis(baseline_state)
optimized_kpis = summarize_kpis(optimized_state)
comparison_df = pd.DataFrame(
    [baseline_kpis, optimized_kpis],
    index=["Baseline", "Coordinated heuristic"],
).round(3)
comparison_df

In [ ]:
# A signed percentage change keeps the direction of each KPI visible.
changes = {}
for metric, baseline_value in baseline_kpis.items():
    optimized_value = optimized_kpis[metric]
    if baseline_value != 0:
        changes[metric] = (
            (optimized_value - baseline_value) / abs(baseline_value) * 100
        )
    else:
        changes[metric] = np.nan

change_df = pd.DataFrame.from_dict(
    changes,
    orient="index",
    columns=["Change: coordinated heuristic vs baseline (%)"],
).round(2)
change_df

## Step 3 — Visualize KPI comparisons

The measures have different units and scales, so this chart uses separate panels rather than placing all measures on one numerical axis. The bars show the values reported in the KPI table; they do not imply that every metric has the same preferred direction.

In [ ]:
def plot_kpi_comparison(
    baseline: Dict[str, float],
    optimized: Dict[str, float],
) -> None:
    """Plot each KPI in its own panel so its unit and scale remain interpretable."""
    metrics = [
        ("total_teu_processed", "Total TEU processed", "TEU"),
        ("avg_crane_utilization", "Average quay-crane utilization", "Fraction"),
        ("avg_yard_utilization", "Average yard utilization proxy", "Fraction"),
        ("avg_gate_queue_trucks", "Average gate queue", "Trucks"),
        (
            "avg_vessel_turnaround_hours",
            "Average completed-vessel turnaround",
            "Hours",
        ),
    ]
    fig, axes = plt.subplots(1, len(metrics), figsize=(16, 4))
    for axis, (metric, title, unit) in zip(axes, metrics):
        values = [baseline[metric], optimized[metric]]
        bars = axis.bar(
            ["Baseline", "Coordinated\nheuristic"],
            values,
            color=["#667085", "#2E90FA"],
        )
        axis.set_title(title)
        axis.set_ylabel(unit)
        axis.grid(True, axis="y", alpha=0.25)
        for bar in bars:
            axis.annotate(
                f"{bar.get_height():.2f}",
                (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=8,
            )
    fig.suptitle("Seven-day simulation: KPI comparison")
    fig.tight_layout()
    plt.show()


plot_kpi_comparison(baseline_kpis, optimized_kpis)

## Step 4 — Time-series visualization of key metrics

A summary table can hide when work was processed or how crane assignments changed. These time series show cumulative TEU processed and the number of currently assigned cranes at each recorded time step. The x-axis is elapsed time from the start of the simulation.

In [ ]:
def plot_timeseries(
    baseline_state: DigitalTwinState,
    optimized_state: DigitalTwinState,
) -> None:
    """Plot cumulative throughput and active crane assignments over time."""
    baseline_df = pd.DataFrame(baseline_state.kpi_history)
    optimized_df = pd.DataFrame(optimized_state.kpi_history)
    for frame in (baseline_df, optimized_df):
        frame["hours_since_start"] = (
            frame["timestamp"] - SIM_START
        ).dt.total_seconds() / 3600

    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    series = [
        (baseline_df, "Baseline", "#667085"),
        (optimized_df, "Coordinated heuristic", "#2E90FA"),
    ]
    for frame, label, color in series:
        axes[0].plot(
            frame["hours_since_start"],
            frame["total_teu_processed"],
            label=label,
            color=color,
        )
        axes[1].plot(
            frame["hours_since_start"],
            frame["active_cranes"],
            label=label,
            color=color,
        )

    axes[0].set_ylabel("Cumulative TEU processed (TEU)")
    axes[0].set_title("Terminal simulation time series")
    axes[1].set_xlabel("Hours since simulation start (h)")
    axes[1].set_ylabel("Assigned quay cranes (count)")
    for axis in axes:
        axis.legend()
        axis.grid(True, alpha=0.25)
    fig.tight_layout()
    plt.show()


plot_timeseries(baseline_state, optimized_state)

## Step 5 — What-if scenario analysis

A scenario changes an input condition while retaining the same simulation rules. This notebook considers three disruptions:

1. **Equipment failure**: two of the six quay cranes are unavailable during the first 12 hours.
2. **Weather delay**: both vessel arrival times are postponed by six hours.
3. **Increased truck variability**: truck arrivals use a normal approximation with the same mean arrivals per step and twice the baseline standard-deviation multiplier.

The baseline and coordinated-heuristic runs use the same scenario inputs and seed. The scenario model does not include weather-dependent crane productivity; the weather case changes vessel arrival times only.

In [ ]:
def run_scenario(
    scenario: str,
    optimized: bool = False,
    seed: int = 42,
) -> DigitalTwinState:
    """Run one disruption scenario with the selected crane-assignment rule."""
    random.seed(seed)
    np.random.seed(seed)

    if scenario == "weather_delay":
        scenario_vessels = [
            Vessel(
                id=vessel.id,
                arrival_time=vessel.arrival_time + timedelta(hours=6),
                teu_total=vessel.teu_total,
                bay_count=vessel.bay_count,
                priority=vessel.priority,
            )
            for vessel in VESSELS
        ]
    else:
        scenario_vessels = VESSELS.copy()

    state = DigitalTwinState(
        current_time=SIM_START,
        vessels=scenario_vessels,
        quay_cranes=QUAY_CRANES.copy(),
        yard_cranes=YARD_CRANES.copy(),
    )
    steps = int(
        timedelta(days=SIM_DURATION_DAYS).total_seconds()
        / TIME_STEP.total_seconds()
    )
    for _ in range(steps):
        elapsed = state.current_time - SIM_START
        if scenario == "equipment_failure" and elapsed < timedelta(hours=12):
            unavailable_cranes = {0, 1}
        else:
            unavailable_cranes = set()

        simulate_step(
            state,
            optimized=optimized,
            unavailable_cranes=unavailable_cranes,
            truck_variability=(scenario == "truck_variability"),
        )
        state.current_time += TIME_STEP
    return state


scenarios = ["equipment_failure", "weather_delay", "truck_variability"]
scenario_results = {}
for scenario in scenarios:
    baseline_scenario_state = run_scenario(
        scenario, optimized=False, seed=42
    )
    optimized_scenario_state = run_scenario(
        scenario, optimized=True, seed=42
    )
    scenario_results[scenario] = {
        "baseline_kpis": summarize_kpis(baseline_scenario_state),
        "optimized_kpis": summarize_kpis(optimized_scenario_state),
    }

scenario_summary = []
for scenario, result in scenario_results.items():
    baseline_turnaround = result["baseline_kpis"][
        "avg_vessel_turnaround_hours"
    ]
    optimized_turnaround = result["optimized_kpis"][
        "avg_vessel_turnaround_hours"
    ]
    if baseline_turnaround != 0:
        turnaround_reduction = (
            (baseline_turnaround - optimized_turnaround)
            / abs(baseline_turnaround)
            * 100
        )
    else:
        turnaround_reduction = np.nan
    scenario_summary.append(
        {
            "scenario": scenario,
            "baseline_turnaround_hours": baseline_turnaround,
            "heuristic_turnaround_hours": optimized_turnaround,
            "turnaround_reduction_percent": turnaround_reduction,
        }
    )

scenario_df = pd.DataFrame(scenario_summary).round(3)
scenario_df

## Step 6 — Interpretation and operational insights

Use the KPI tables and plots to make claims about this simulated instance. A positive change in throughput or utilization means that the coordinated heuristic's reported value is higher; for queue size and turnaround, a lower value is usually preferable. A zero turnaround value can mean that no vessel completed within the horizon, so it should not be interpreted as zero hours of service. Check the completion count and the modeled assumptions before comparing such averages.

Scenario results show how the rules respond to the specific disruptions encoded here. They do not establish that the heuristic will improve a real terminal: that conclusion would require calibrated data, validation, and operational constraints beyond this model. No annual cost savings are calculated because the notebook contains no cost inputs.

The central lesson is methodological: a system-level simulation can connect subsystem assumptions to shared performance measures, while the credibility of its conclusions depends on the quality and scope of those assumptions.

## Step 7 — Why this Tier exists relative to Tier 4

### Advantages of a system-level simulation
- It tracks quay, yard, vessel, and gate measures within one shared time horizon.
- It supports controlled what-if comparisons by changing a defined input condition.
- It makes interactions and modeling assumptions explicit enough to inspect and discuss.

### Limitations
- A multi-module model requires data integration and calibration before its outputs can support operational decisions.
- Fine time steps and detailed subsystem rules increase computational and development effort.
- Simplified proxies, such as the aggregate yard utilization used here, cannot replace detailed operational data.

### When this approach is useful
- For exploring how specified assumptions affect system-level KPIs.
- For structuring disruption studies and identifying which operational data are needed.
- As a foundation for model validation and for adding realistic constraints in later work.

This tier emphasizes the simulation of interacting terminal subsystems. Its crane coordination rule remains the transparent priority-based heuristic defined in this notebook; it should not be confused with a learned policy or an exact scheduling optimization model.